In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

options = ['A', 'B', 'C', 'D', 'E']

In [2]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

print("Train shape:", train.shape)
print("Test shape :", test.shape)

Train shape: (2000, 8)
Test shape : (500, 7)


In [3]:
def combine_prompt_option(row, option):
    return row['prompt'] + "  |  " + row[option]

def map_at_3(df, predict_fn):
    scores = []
    for _, row in df.iterrows():
        prediction = predict_fn(row)
        predicted_labels = prediction.split()
        correct = row['answer']
        score = 0.0
        if correct in predicted_labels:
            rank = predicted_labels.index(correct) + 1
            score = 1.0 / rank
        scores.append(score)
    return np.mean(scores)

In [4]:
def predict_top3_tfidf(row):
    texts = [combine_prompt_option(row, opt) for opt in options]
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(texts)
    prompt_vec = vectorizer.transform([row['prompt']])
    scores = cosine_similarity(prompt_vec, tfidf_matrix).flatten()
    top3_indices = scores.argsort()[::-1][:3]
    return ' '.join([options[i] for i in top3_indices])

In [5]:
sample = train.sample(200, random_state=42)
score = map_at_3(sample, predict_top3_tfidf)
print(f"TF-IDF Local MAP@3: {score:.4f}")

TF-IDF Local MAP@3: 0.2858


In [ ]:
test['Prediction'] = test.apply(predict_top3_tfidf, axis=1)
submission = test[['id', 'Prediction']].copy()
submission.columns = ['ID', 'Prediction']
submission.to_csv('tfidf_submission.csv', index=False)
print(submission.head())
# Kaggle MAP@3: 0.29260 (V11)

   ID Prediction
0   1      A D E
1   2      C A D
2   3      A D C
3   4      A E C
4   5      C A D
